# Aadhaar KYC Face Matching Pipeline - SageMaker Setup

**Instance type:** `ml.g5.xlarge` (A10G 24GB) or `ml.g4dn.xlarge` (T4 16GB) minimum

This notebook sets up and runs the pipeline on SageMaker with locally downloaded HuggingFace models.

## Step 1: Clone the repo

In [ ]:
!git clone https://github.com/MlvPrasadOfficial/GENAI_AADHAR.git
%cd GENAI_AADHAR

## Step 2: Install dependencies
Order matters — do NOT rearrange.

In [ ]:
# Check GPU availability first
!nvidia-smi

In [ ]:
# Install in order (order matters for compatibility)
!pip install torch==2.3.1+cu121 torchvision==0.18.1+cu121 --extra-index-url https://download.pytorch.org/whl/cu121
!pip install numpy==1.26.4
!pip install onnxruntime-gpu==1.19.2
!pip install basicsr==1.4.2
!pip install realesrgan==0.3.0
!pip install insightface==0.7.3
!pip install opencv-python==4.10.0.84 Pillow==10.4.0 PyYAML==6.0.2 requests==2.32.3 tqdm==4.66.4 PyMuPDF==1.24.5

In [ ]:
# Verify GPU is available for all frameworks
import torch
print(f"PyTorch CUDA: {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'})")

import onnxruntime
print(f"ONNX Runtime providers: {onnxruntime.get_available_providers()}")

## Step 3: Setup model weights

The pipeline expects models in the `models/` directory. If your company has them downloaded from HuggingFace, copy or symlink them.

In [ ]:
# Option A: Download models automatically (if internet available)
!python scripts/download_models.py

In [ ]:
# Option B: Copy from your HuggingFace cache or S3
# Uncomment and adjust paths to match your setup:

# --- From S3 ---
# !aws s3 cp s3://your-bucket/models/insightface/ models/insightface/ --recursive
# !aws s3 cp s3://your-bucket/models/realesrgan/ models/realesrgan/ --recursive

# --- From local HuggingFace cache ---
# !cp -r /home/ec2-user/.cache/huggingface/hub/models--buffalo_l/ models/insightface/models/buffalo_l/
# !cp /path/to/RealESRGAN_x4plus.pth models/realesrgan/

# --- From SageMaker EFS mount ---
# !ln -s /efs/shared-models/insightface models/insightface
# !ln -s /efs/shared-models/realesrgan models/realesrgan

In [ ]:
# Verify model files are in place
!echo "=== InsightFace buffalo_l ==="
!ls -la models/insightface/models/buffalo_l/ 2>/dev/null || echo "NOT FOUND - need buffalo_l model pack"

!echo "\n=== Real-ESRGAN ==="
!ls -la models/realesrgan/ 2>/dev/null || echo "NOT FOUND - need RealESRGAN_x4plus.pth"

## Step 4: Configure for SageMaker

The VLM guard (Ollama) is not available on SageMaker. We disable it and rely on cosine + multi-metric scoring.

In [ ]:
import yaml

with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Disable VLM guard (no Ollama on SageMaker)
config['vlm_guard']['enabled'] = False

# Ensure GPU is used
config['face']['ctx_id'] = 0  # GPU device 0

# Enable parallel batch for faster processing
config['batch']['parallel'] = True
config['batch']['max_workers'] = 4

with open('config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("Config updated for SageMaker:")
print(f"  VLM guard: {config['vlm_guard']['enabled']}")
print(f"  GPU device: {config['face']['ctx_id']}")
print(f"  Parallel batch: {config['batch']['parallel']}")
print(f"  Workers: {config['batch']['max_workers']}")

## Step 5: Upload test images

Upload your Aadhaar cards and selfies to the `FILES/` directory.

In [ ]:
# Create directories
!mkdir -p FILES/AADHAR FILES/SELFIE

# Option A: Copy from S3
# !aws s3 cp s3://your-bucket/test-data/AADHAR/ FILES/AADHAR/ --recursive
# !aws s3 cp s3://your-bucket/test-data/SELFIE/ FILES/SELFIE/ --recursive

# Option B: Upload via Jupyter file browser (drag and drop into FILES/AADHAR/ and FILES/SELFIE/)

# Verify
!echo "Aadhaar files:" && ls FILES/AADHAR/ 2>/dev/null
!echo "Selfie files:" && ls FILES/SELFIE/ 2>/dev/null

## Step 6: Run the pipeline

In [ ]:
# Run unit tests first to verify setup
!python -m pytest tests/ -v -m "not integration" --tb=short 2>&1 | tail -20

In [ ]:
# Single pair test
!python main.py --aadhaar FILES/AADHAR/AADHAR001.jpg --selfie FILES/SELFIE/USER_01.jpg --verbose

In [ ]:
# Batch mode - all 90 pairs
!python main.py --batch pairs.csv --verbose

In [ ]:
# JSON output for programmatic use
import subprocess, json

result = subprocess.run(
    ['python', 'main.py', '--aadhaar', 'FILES/AADHAR/AADHAR001.jpg', 
     '--selfie', 'FILES/SELFIE/USER_01.jpg', '--json-output'],
    capture_output=True, text=True
)

data = json.loads(result.stdout)
print(f"Match: {data['match']}")
print(f"Confidence: {data['confidence_pct']}%")
print(f"Cosine: {data['cosine_score']}")

## Step 7: Use as Python library (no CLI)

In [ ]:
from utils.config_loader import load_config
from pipeline.orchestrator import KYCPipelineOrchestrator

# Load once
config = load_config('config.yaml')
pipeline = KYCPipelineOrchestrator(config)
pipeline.load()

# Run on any pair
aadhaar_bytes = open('FILES/AADHAR/AADHAR001.jpg', 'rb').read()
selfie_bytes = open('FILES/SELFIE/USER_01.jpg', 'rb').read()

result = pipeline.run(aadhaar_bytes, selfie_bytes)

print(f"Match: {result.match}")
print(f"Confidence: {result.confidence_pct}%")
print(f"Cosine: {result.cosine_score:.4f}")
print(f"Fused: {result.fused_score:.4f}")
print(f"Aadhaar: {result.aadhaar_gender} age {result.aadhaar_age}")
print(f"Selfie: {result.selfie_gender} age {result.selfie_age}")

## Step 8: Save results to S3 (optional)

In [ ]:
# Upload logs and results to S3
# !aws s3 cp logs/ s3://your-bucket/kyc-results/logs/ --recursive
# print("Results uploaded to S3")